<p align="center">
  <img src="https://img.shields.io/badge/Research%20Mode-ON-4cbb17?style=for-the-badge" alt="Research Mode">
</p>

# Assembling CRN Metadata

Builds one cleaned, sample-level metadata table for a CRN Cloud cohort by merging its SAMPLE, SUBJECT, and CLINPATH tables. Use the output as the starting point for any analysis that needs sample, subject, and clinical metadata in one place.

**To use another cohort**, change only the parameters block in Section 1. The notebook works with cohorts that have SAMPLE, SUBJECT, and CLINPATH tables (human postmortem and clinical cohorts).

**Inputs** (the cohort's `metadata/` folder)
- `SAMPLE.csv`: one row per sample
- `SUBJECT.csv`: demographics per subject
- `CLINPATH.csv`: clinical and neuropathology data per subject

**Outputs** (`ws_files/crn_metadata/<cohort>/`)
- `<cohort>_cleanedmetadata.csv`: one row per sample per dataset
- `subject_conflicts.csv`, `clinpath_conflicts.csv`: fields where duplicate records disagreed, with the value that was kept
- `clinpath_selection_ties.csv`: subjects with more than one equally complete record

**Key decisions**
- Rows are unique on `ASAP_dataset_id` + `ASAP_sample_id`, because a sample can appear in more than one dataset.
- SUBJECT is deduplicated within each dataset. CLINPATH is deduplicated per person (`ASAP_subject_id`), because the same person can be submitted under more than one dataset.
- Duplicate records that fill in different fields are combined into one complete record.
- When duplicate records disagree, the value from the most complete record is kept and the disagreement is logged. No values are changed silently.
- Blank strings and missing labels, including `Unknown` and `Not Reported`, become empty values (see Section 3).

**Tested with:** CRN Cloud v5.0.0, `cohort-pmdbs-sc-rnaseq`

## 1. Setup
Set the CRN release and paths. Outputs go to `ws_files`, which persists across apps and sessions.

In [ ]:
from pathlib import Path
from typing import Sequence
import numpy as np
import pandas as pd
HOME = Path.home()

# --- Set these for your cohort ---
COLLECTION = "ASAPCRN_PMDBS_scRNAseq"       # resource folder in the workspace
DATASET    = "pmdbs-sc-rnaseq-v3"
COHORT     = "cohort-pmdbs-sc-rnaseq"
RELEASE    = "v5.0.0"                       # CRN Cloud release, for your records
# ---------------------------------

WS_ROOT = Path.home() / "workspace"
CRN_METADATA_DIR = WS_ROOT / COLLECTION / DATASET / COHORT / "metadata"
OUTPUT_DIR = WS_ROOT / "ws_files" / "crn_metadata" / COHORT
OUTPUT_PATH = OUTPUT_DIR / f"{COHORT}_cleanedmetadata.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("CRN release:", RELEASE)
print("Metadata:   ", CRN_METADATA_DIR)
print("Outputs:    ", OUTPUT_DIR)

## 2. Load Metadata Tables
Load the three release tables and check their sizes. SAMPLE sets the row count for the final table: every merge below must keep exactly one row per sample.

In [ ]:
SAMPLE = pd.read_csv(CRN_METADATA_DIR / "SAMPLE.csv", low_memory=False)
SUBJECT = pd.read_csv(CRN_METADATA_DIR / "SUBJECT.csv", low_memory=False)
CLINPATH = pd.read_csv(CRN_METADATA_DIR / "CLINPATH.csv", low_memory=False)

expected_sample_rows = len(SAMPLE)

print("SAMPLE:  ", SAMPLE.shape)
print("SUBJECT: ", SUBJECT.shape)
print("CLINPATH:", CLINPATH.shape)

## 3. Helper Functions

**`standardize_missing`** converts blank strings and common missing labels (`NA`, `N/A`, `NULL`, `NONE`, [`UNKNOWN`, `NOT REPORTED`]) to `pd.NA`, so "missing" means the same thing in every table.

> ⚠️ [If you keep UNKNOWN / NOT REPORTED in the list:] In clinical fields, "Unknown" can be a real answer (for example, family history unknown). This notebook treats it as missing. Remove those labels from `MISSING_LABELS` if you need to tell them apart.

**`collapse_duplicate_records`** reduces duplicate records to one row per key:
1. The row with the most filled-in fields is used as the base.
2. If a field has one value across the duplicates, that value is filled in.
3. If a field has more than one value, the base row's value is kept and all observed values are logged in the conflict report.
4. If several rows are equally complete, the first is used and the tie is logged.

Audit columns added to each row: `_source_record_count`, `_selection_tie`, `_n_conflicting_fields`, `_conflicting_fields`.

In [ ]:
MISSING_LABELS = {
    "",
    "NA",
    "N/A",
    "NAN",
    "NONE",
    "NULL",
    "NOT REPORTED",
    "UNKNOWN",
}


def standardize_missing(data: pd.DataFrame) -> pd.DataFrame:
    """Convert blank strings and common missing labels to pd.NA."""
    result = data.copy()
    text_columns = result.select_dtypes(
        include=["object", "string"]
    ).columns
    for column in text_columns:
        cleaned = result[column].astype("string").str.strip()
        missing_mask = cleaned.str.upper().isin(MISSING_LABELS)
        result[column] = cleaned.mask(missing_mask, pd.NA)
    return result


def collapse_duplicate_records(
    data: pd.DataFrame,
    keys: Sequence[str],
    table_name: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Collapse duplicate keys while preserving agreed complementary values.
    Selection and fill rules:
    1. Select the row with the most populated non-key fields as the base.
    2. If a field has exactly one distinct non-null value in the group,
       place that value in the collapsed row.
    3. If a field has multiple distinct values, retain the selected base
       value and record all observed values in the conflict report.
    4. Record ties when multiple rows are equally complete.
    """
    work = standardize_missing(data)
    metadata_columns = [
        column for column in work.columns
        if column not in keys
    ]
    work["_fields_present"] = (
        work[metadata_columns]
        .notna()
        .sum(axis=1)
    )
    work["_source_row"] = np.arange(len(work))
    selected_records = []
    conflict_records = []
    tie_records = []
    group_key = keys[0] if len(keys) == 1 else list(keys)
    for key_values, group in work.groupby(
        group_key,
        dropna=False,
        sort=False,
    ):
        if not isinstance(key_values, tuple):
            key_values = (key_values,)
        key_record = dict(zip(keys, key_values))
        max_fields = int(group["_fields_present"].max())
        top_rows = (
            group.loc[
                group["_fields_present"].eq(max_fields)
            ]
            .sort_values("_source_row")
        )
        selected = top_rows.iloc[0].copy()
        if len(top_rows) > 1:
            tie_records.append({
                **key_record,
                "table": table_name,
                "n_tied_rows": len(top_rows),
                "fields_present": max_fields,
                "source_rows": " | ".join(
                    top_rows["_source_row"]
                    .astype(str)
                    .tolist()
                ),
            })
        conflicting_fields = []
        for column in metadata_columns:
            values = (
                group[column]
                .dropna()
                .drop_duplicates()
            )
            if len(values) == 1:
                selected[column] = values.iloc[0]
            elif len(values) > 1:
                conflicting_fields.append(column)
                selected_top_values = (
                    top_rows[column]
                    .dropna()
                )
                if len(selected_top_values):
                    selected[column] = (
                        selected_top_values.iloc[0]
                    )
                elif pd.isna(selected[column]):
                    selected[column] = values.iloc[0]
                conflict_records.append({
                    **key_record,
                    "table": table_name,
                    "field": column,
                    "n_unique_values": len(values),
                    "observed_values": " | ".join(
                        repr(value)
                        for value in values.tolist()
                    ),
                    "selected_value": repr(
                        selected[column].item()
                        if isinstance(selected[column], np.generic)
                        else selected[column]
                    ),
                    "n_source_records": len(group),
                })
        selected["_source_record_count"] = len(group)
        selected["_selection_tie"] = len(top_rows) > 1
        selected["_n_conflicting_fields"] = len(
            conflicting_fields
        )
        selected["_conflicting_fields"] = (
            " | ".join(conflicting_fields)
            if conflicting_fields
            else pd.NA
        )
        selected_records.append(selected)
    collapsed = (
        pd.DataFrame(selected_records)
        .drop(
            columns=[
                "_fields_present",
                "_source_row",
            ]
        )
        .reset_index(drop=True)
    )
    conflicts = pd.DataFrame(conflict_records)
    ties = pd.DataFrame(tie_records)
    if collapsed.duplicated(list(keys)).any():
        raise AssertionError(
            f"{table_name} cleaning did not produce "
            "one row per key."
        )
    return collapsed, conflicts, ties

## 4. Merge SAMPLE with SUBJECT
SUBJECT is deduplicated on `ASAP_subject_id` + `ASAP_dataset_id` + `subject_id`, then left-joined onto SAMPLE.
- Only the SUBJECT columns this cohort has are used.
- `validate="many_to_one"` makes sure each sample matches at most one subject record, and the row count check confirms no samples were added or lost.
- Columns SAMPLE already has aren't added again, so there are no duplicate `_x` / `_y` columns.

In [ ]:
subject_keys = ["ASAP_subject_id", "ASAP_dataset_id", "subject_id"]
subject_columns = subject_keys + [
    "source_subject_id", "sex", "race", "ethnicity", "biobank_name", "gp2_phenotype",
]
# Keep only the columns this cohort's SUBJECT table has
subject_columns = [c for c in subject_columns if c in SUBJECT.columns]

subject_clean, subject_conflicts, subject_selection_ties = collapse_duplicate_records(
    SUBJECT[subject_columns], subject_keys, table_name="SUBJECT"
)
print("SUBJECT rows:", len(SUBJECT), "->", len(subject_clean))
print("SUBJECT conflicts:", len(subject_conflicts))
print("SUBJECT ties:", len(subject_selection_ties))

# Join onto samples, leaving out audit columns and columns SAMPLE already has
already = [c for c in subject_clean.columns if c in SAMPLE.columns and c not in subject_keys]
if already:
    print("Already in SAMPLE, not added again:", already)
subject_payload = [c for c in subject_clean.columns if not c.startswith("_") and c not in already]

df = SAMPLE.merge(
    subject_clean[subject_payload], on=subject_keys,
    how="left", validate="many_to_one", indicator="_subject_merge",
)
assert len(df) == expected_sample_rows, "SUBJECT merge changed the SAMPLE row count."

print("\nSUBJECT merge coverage:")
print(df["_subject_merge"].value_counts())
df = df.drop(columns="_subject_merge")

## 5. Merge with CLINPATH

### 5.1 Check CLINPATH duplicates
Some subjects have several CLINPATH rows within the same dataset. Before collapsing, check whether these are:
- **Fragments**: rows that each hold part of the record, differing only in which fields are filled. These are safe to combine.
- **A real dimension**: rows that differ in every case on one field (for example, visit or instrument). Collapsing these would lose information.

If no field differs across every duplicate group, the rows are fragments.

In [ ]:
# Count rows per subject within each dataset, then check whether any field
# differs inside the duplicate groups. Fields listed below are the ones where
# duplicate rows disagree; they will show up in the conflict report.

clin_key = ["ASAP_dataset_id", "ASAP_subject_id"]
clin_grp = CLINPATH.groupby(clin_key)

clin = standardize_missing(CLINPATH)
print("Rows per subject within a dataset:")
print(clin.groupby(clin_key).size().value_counts().sort_index(), "\n")

clin_dupe = clin_grp.filter(lambda g: len(g) > 1)
clin_value_cols = [c for c in CLINPATH.columns if c not in clin_key + ["ASAP_team_id",
              "subject_id", "source_subject_id"]]
per_group_max_distinct = (clin_dupe.groupby(clin_key)[clin_value_cols].nunique(dropna=True).max()
                              .sort_values(ascending=False))

print("Max distinct values any duplicate group shows, per field (top 8):")
print(per_group_max_distinct.head(8))
print("\n-> No column distinct in *every* row = fragmentation, safe to collapse.")

# Surface the odd-count subject explicitly 
odd = clin_grp.filter(lambda g: len(g) == 2)
if len(odd):
    print("\nOdd-count (2-row) subject — inspect directly:")
    display(odd.T)

### 5.2 Collapse CLINPATH to one row per person
CLINPATH is collapsed on `ASAP_subject_id` alone. This handles fragments within a dataset and the same person submitted under more than one dataset in one pass, so it can find conflicts the check above doesn't.

Dataset, team, and local ID columns are dropped before collapsing. They differ by design when a person is in more than one dataset, so they would show up as false conflicts, and each sample keeps its own IDs from SAMPLE anyway.

**Vocabulary maps:** `VOCAB_MAPS` merges synonyms before collapsing, so they don't count as conflicts (for example, `"No PD nor other neurological disorder"` and `"Healthy Control"` in `primary_diagnosis`). It's empty by default. Add a mapping only after it's confirmed with the data team.

In [ ]:
# Synonyms to merge before collapsing, as {column: {old_value: new_value}}.
# Example: {"primary_diagnosis": {"No PD nor other neurological disorder": "Healthy Control"}}
VOCAB_MAPS = {}

CLINPATH_h = CLINPATH.copy()
for col, mapping in VOCAB_MAPS.items():
    if col in CLINPATH_h.columns:
        CLINPATH_h[col] = CLINPATH_h[col].replace(mapping)

# Collapse to one row per person. Dataset, team, and local ID columns are dropped
# first: they differ by design across datasets and would show up as false conflicts.
id_cols = ["ASAP_dataset_id", "ASAP_team_id", "subject_id", "source_subject_id"]
CLINPATH_clean, CLINPATH_conflicts, CLINPATH_ties = collapse_duplicate_records(
    CLINPATH_h.drop(columns=id_cols, errors="ignore"), ["ASAP_subject_id"], table_name="CLINPATH"
)

clin_multi = CLINPATH_clean[CLINPATH_clean["_source_record_count"] > 1]
complementary = clin_multi[clin_multi["_n_conflicting_fields"] == 0]
conflicted = clin_multi[clin_multi["_n_conflicting_fields"] > 0]

print(f"Person-level rows:              {len(CLINPATH_clean)}")
print(f"Subjects collapsed from >1 row: {len(clin_multi)}")
print(f"  - combined, no conflicts:     {len(complementary)}")
print(f"  - with conflicts to review:   {len(conflicted)}")

### 5.3 Example: rebuilding a fragmented record
One subject whose duplicate rows each held different fields. BEFORE shows only the fields filled in some rows and empty in others. AFTER shows the single combined record.

In [ ]:
# Pick the first subject whose duplicates combined with no conflicts,
# and show only the fields that were split across its rows.
if complementary.empty:
    print("No fragmented records to show.")
else:
    ex_id = complementary["ASAP_subject_id"].iloc[0]
    raw = standardize_missing(CLINPATH_h[CLINPATH_h["ASAP_subject_id"] == ex_id])
    frag_cols = [c for c in clin_value_cols if raw[c].notna().any() and raw[c].isna().any()]

    print(f"BEFORE: {ex_id} in {len(raw)} rows")
    display(raw[["ASAP_dataset_id"] + frag_cols].T)
    print("AFTER: one combined record")
    display(CLINPATH_clean.loc[CLINPATH_clean["ASAP_subject_id"] == ex_id, frag_cols].T)

### 5.4 Join CLINPATH onto samples
Only the clinical fields are joined, on `ASAP_subject_id`. The ID columns were dropped in 5.2, so a person-level record can't overwrite each sample's own dataset context. Audit columns and columns already in the table aren't added either.

In [ ]:
# Leave out audit columns and anything already in the table
already = [c for c in CLINPATH_clean.columns if c in df.columns and c != "ASAP_subject_id"]
if already:
    print("Already in the table, not added again:", already)
clin_payload = [c for c in CLINPATH_clean.columns if not c.startswith("_") and c not in already]

df = df.merge(
    CLINPATH_clean[clin_payload], on="ASAP_subject_id",
    how="left", validate="many_to_one", indicator="_clinpath_merge",
)
assert len(df) == expected_sample_rows, "CLINPATH merge changed the row count."

print("CLINPATH merge coverage:")
print(df["_clinpath_merge"].value_counts())
df = df.drop(columns="_clinpath_merge")

 ## 6. Final Cleanup
- SAMPLE columns were merged without cleaning, so missing values are standardized once across the whole table.
- Columns that are empty for every sample are dropped. This list can change between releases, so the dropped columns are printed.

In [ ]:
# SAMPLE columns were merged without cleaning, so standardize missing values once here
df = standardize_missing(df)

# Drop columns that are empty for every sample
empty_cols = df.columns[df.isna().all()].tolist()
df = df.drop(columns=empty_cols)
print(f"Dropped {len(empty_cols)} empty columns:", empty_cols)
print("Final table:", df.shape)
df.head()

## 7. Save Outputs and Summary
Save the cleaned table and the conflict reports, then print a summary. Check the conflict reports before using the disputed fields in an analysis.

In [ ]:
# Final checks
assert len(df) == expected_sample_rows, "Final table doesn't match SAMPLE row count."

# GP2 linkage coverage: GP2_id is the bridge to GP2 genetics data
n_gp2 = df["GP2_id"].notna().sum() if "GP2_id" in df else 0

print("=== Summary ===")
print(f"Samples:            {len(df)}")
print(f"Subjects:           {df['ASAP_subject_id'].nunique()}")
print(f"Columns:            {df.shape[1]}")
print(f"SUBJECT conflicts:  {len(subject_conflicts)}")
print(f"CLINPATH conflicts: {len(CLINPATH_conflicts)} fields across {len(conflicted)} subjects")
print(f"CLINPATH ties:      {len(CLINPATH_ties)}")
print(f"Samples with GP2_id: {n_gp2} / {len(df)}")

In [ ]:
df.to_csv(OUTPUT_PATH)

## 8. Extra Investigation of Conflicts
Section 7 counts the CLINPATH conflicts. This section shows each one next to the values each dataset submitted, and sorts them by type so you know what to do with each:

| Type | Meaning | What to do |
|---|---|---|
| **Formatting only** | Same value written differently (case, spacing, `85` vs `85.0`) | Safe to ignore, or add the variants to `VOCAB_MAPS` |
| **Real disagreement, between datasets** | The same person was submitted with different values in two datasets | Synonyms go in `VOCAB_MAPS` once confirmed; true differences go to the data team |
| **Real disagreement, within one dataset** | One team submitted rows that disagree | Likely a data entry issue: report to the data team |

The results are saved to `clinpath_conflicts_review.csv`. It contains subject IDs and clinical values, so share it within the program only.

In [ ]:
# Look at each conflict next to the values each dataset submitted
clin_std = standardize_missing(CLINPATH_h)

def normalize(v):
    """Treat '85', 85 and 85.0 as the same, and ignore case and extra spaces."""
    try:
        return float(v)
    except (TypeError, ValueError):
        return " ".join(str(v).lower().split())

rows = []
for _, r in CLINPATH_conflicts.iterrows():
    subj, field = r["ASAP_subject_id"], r["field"]
    vals = (clin_std.loc[clin_std["ASAP_subject_id"] == subj, ["ASAP_dataset_id", field]]
            .dropna().drop_duplicates())
    within_one_dataset = vals.groupby("ASAP_dataset_id")[field].nunique().max() > 1
    rows.append({
        "ASAP_subject_id": subj,
        "field": field,
        "values_by_dataset": "; ".join(f"{d}: {v}" for d, v in vals.itertuples(index=False)),
        "kept": r["selected_value"],
        "type": ("formatting only" if vals[field].map(normalize).nunique() == 1
                 else "real disagreement"),
        "where": "within one dataset" if within_one_dataset else "between datasets",
    })

review = pd.DataFrame(rows)
review.to_csv(OUTPUT_DIR / "clinpath_conflicts_review.csv", index=False)

display(pd.crosstab(review["type"], review["where"], margins=True))
display(review["field"].value_counts().rename("conflicts").to_frame())
display(review.sort_values(["type", "field", "ASAP_subject_id"]))

## 9. Summary and Next Steps

**What this notebook produced** (in `ws_files/crn_metadata/<cohort>/`)
- `<cohort>_cleanedmetadata.csv`: one row per sample per dataset, with SAMPLE, SUBJECT, and CLINPATH fields
- `subject_conflicts.csv`, `clinpath_conflicts.csv`, `clinpath_selection_ties.csv`: where duplicate records disagreed or tied, and what was kept
- `clinpath_conflicts_review.csv`: CLINPATH conflicts sorted by type

**Next steps**
1. **Review the conflicts** in `clinpath_conflicts_review.csv`, starting with the fields you plan to analyze.
2. **Add confirmed synonyms** to `VOCAB_MAPS` (Section 5.2) and rerun the notebook. Those conflicts will disappear.
3. **Report real disagreements** to the data team or [cloud@parkinsonsroadmap.org](mailto:cloud@parkinsonsroadmap.org).
4. **Rerun for each new CRN Cloud release.** Update `RELEASE` in Section 1, and check the dropped-columns list in Section 6, since it can change.

**Using the cleaned table**
```python
meta = pd.read_csv(OUTPUT_PATH)
```
Join it to other data on `ASAP_dataset_id` + `ASAP_sample_id`, not the sample ID alone, because a sample can appear in more than one dataset. Count people with `ASAP_subject_id`, not rows.

**Related tutorials**
- [GP2 × CRN Interoperability](../GP2_CRN_Interoperability/): links these samples to GP2 participants and their genetic data